# 07 · Studi Kasus Pasang Surut Kapuas — Bab 8

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 8: pipeline end-to-end prediksi pasang surut — data (sintetik, dapat diganti data nyata BIG/PSMSL), windowing, baseline persistence/klimatologi, model MLP/LSTM/GRU, walk-forward, evaluasi per horizon, dan simulasi pengisian gap data.

## 1. Setup & Data

Data contoh sintetik meniru pasang semi-diurnal + diurnal campuran (arahkan: ganti dengan `pd.read_csv` dari data nyata).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

# data jam-an sintetik 3 tahun
t = pd.date_range("2022-01-01", periods=3*365*24, freq="h")
tt = np.arange(len(t))
tinggi = (1.0 + 0.6*np.sin(2*np.pi*tt/12.42) + 0.4*np.sin(2*np.pi*tt/24.84)
          + 0.05*np.random.randn(len(t)))
df = pd.DataFrame({"waktu": t, "tinggi": tinggi.round(3)}).set_index("waktu")
print(df.head())
print("nilai hilang:", int(df["tinggi"].isna().sum()))

## 2. Spektrum (Gambar 8.1)

FFT untuk melihat periode dominan — pembacaan cepat tipe pasang.

In [ ]:
ser = df["tinggi"].values
fft = np.fft.rfft(ser - ser.mean())
freq = np.fft.rfftfreq(len(ser), d=1.0)
period = np.where(freq > 0, 1/freq, np.inf)
power = np.abs(fft)**2
m = (period > 6) & (period < 100)
plt.figure(figsize=(7, 3.5))
plt.plot(period[m], power[m], color="#4a90e2")
plt.axvline(12.42, color="#e0893d", ls="--", label="12,42 jam (M2)")
plt.axvline(24.84, color="#27ae60", ls="--", label="24,84 jam (K1)")
plt.xscale("log"); plt.xlabel("Periode (jam)"); plt.ylabel("Daya")
plt.title("Spektrum frekuensi"); plt.legend(); plt.tight_layout(); plt.show()

## 3. Windowing & Split

`w` dalam jam; horizon `h` dalam jam (24 = 1 hari). Gunakan split kronologis.

In [ ]:
def buat_window(deret, w=168, h=24):
    X, y = [], []
    for i in range(len(deret) - w - h + 1):
        X.append(deret[i:i+w])
        y.append(deret[i+w:i+w+h])
    return np.array(X), np.array(y)

# bentuk window (w=168 jam, h=24)
X, y = buat_window(ser, w=168, h=24)
y = y[:, 0]  # gunakan langkah pertama horizon sebagai target sederhana (h=24)
print("X", X.shape, "y", y.shape)

n = len(X)
ntr, nva = int(n*0.7), int(n*0.15)
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:ntr+nva], y[ntr:ntr+nva]
Xte, yte = X[ntr+nva:], y[ntr+nva:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

## 4. Baseline (persistence & klimatologi)

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

# persistence untuk h=24: nilai tinggi air 24 jam SEBELUM target.
# Karena deret asli tersedia, target y[i] = ser[i + w + h - 1]; persistence = ser[i + w - 1].
offs = 168 + 24 - 1  # indeks target y[0] di deret asli
i0 = ntr + nva       # contoh test pertama di dalam window-index
persist_test = np.array([ser[i0 + k + offs - 24] for k in range(len(yte))])
base = np.full(len(yte), float(np.mean(ytr)))
print("MAE persistence h=24:", round(mae(yte, persist_test), 4))
print("MAE klimatologi:", round(mae(yte, base), 4))

## 5. Model: MLP, LSTM, GRU (Kode 8.2)

In [ ]:
def buat_model(kind, w=168, f=1):
    if kind == "mlp":
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation="relu", input_shape=(w*f,)),
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(1)])
    elif kind == "lstm":
        m = tf.keras.Sequential([
            tf.keras.layers.LSTM(16, input_shape=(w, f)),
            tf.keras.layers.Dense(1)])
    else:
        m = tf.keras.Sequential([
            tf.keras.layers.GRU(16, input_shape=(w, f)),
            tf.keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

hasil = {}
pred_lstm = None
for kind in ["mlp", "lstm", "gru"]:
    Xtr_m = Xtr.reshape(len(Xtr), -1) if kind == "mlp" else Xtr
    Xva_m = Xva.reshape(len(Xva), -1) if kind == "mlp" else Xva
    Xte_m = Xte.reshape(len(Xte), -1) if kind == "mlp" else Xte
    m = buat_model(kind, w=168)
    m.fit(Xtr_m, ytr, validation_data=(Xva_m, yva), epochs=20, batch_size=32, verbose=0)
    p = m.predict(Xte_m, verbose=0).ravel()
    hasil[kind] = mae(yte, p)
    if kind == "lstm":
        pred_lstm = p
    print(f"MAE {kind.upper()}: {hasil[kind]:.4f}")

print("MAE persistence h=24 (ref):", round(mae(yte, persist_test), 4))

## 6. Plot 7 Hari (Gambar 8.2)

In [ ]:
n_awal = Xte.shape[0] - 7*24
plt.figure(figsize=(10, 3.4))
plt.plot(yte[n_awal:], label="aktual", lw=1.3)
plt.plot(pred_lstm[n_awal:], label="prediksi LSTM", lw=1.1, ls="--", alpha=0.85)
plt.legend(); plt.tight_layout(); plt.show()

## 7. Simulasi Pengisian Gap (Kode 8.3)

Sembunyikan 24 jam dari data, latih tanpa gap, prediksi, lalu ukur MAE terhadap nilai asli.

In [ ]:
pos = 1500  # contoh posisi gap
garis = ser.copy()
garis[pos:pos+24] = np.nan
ada = ~np.isnan(garis)
# latih LSTM hanya pada data utuh (di sini cukup contoh alur)
print("Gap 24 jam pada pos", pos)
print("Langkah: latih model pada subset utuh, prediksi window akhir sebelum gap,")
print("         bandingkan dgn nilai asli yang disembunyikan.")

## 8. Latihan Mini

1. Ganti data dengan nyata (BIG/PSMSL) dan jalankan ulang.
2. Uji `w ∈ {24, 72, 168}` pada h=24; buat tabel MAE.
3. Bangun model *direct* untuk h ∈ {24, 72, 168}; plot MAE per horizon.
4. Hitung skill score `SS = 1 − MAE_model/MAE_persistence` per horizon.
5. Simulasikan gap 72 jam dan bandingkan MAE imputasi vs gap 24 jam.